# Install Libraries

In [ ]:
!pip install fastapi nest-asyncio uvicorn transformers torch sentencepiece omegaconf requests protobuf pydantic sacremoses

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.2 MB 2.4 MB/s eta 0:00:05
   ----- ---------------------------------- 1.3/10.2 MB 2.4 MB/s eta 0:00:04
   -------- ------------------------------- 2.1/10.2 MB 2.7 MB/s eta 0:00:04
   --------- ------------------------------ 2.4/10.2 MB 2.5 MB/s eta 0:00:04
   ---------- ----------------------------- 2.6/10.2 MB 2.5 MB/s eta 0:00:04
   ------------- -------------------------- 3.4/10.2 MB 2.4 MB/s eta 0:00:03
   -------------- ------------------------- 3.7/10.2 MB 2.4 MB/s eta


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Create Config File

In [3]:
yaml_config = """
model_path: "Helsinki-NLP/opus-mt-en-vi"
"""
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

# Quick Model Test

In [3]:
from transformers import MarianMTModel, MarianTokenizer

MODEL_NAME = "Helsinki-NLP/opus-mt-en-vi"
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model     = MarianMTModel.from_pretrained(MODEL_NAME)

text = "Hello, how are you?"
inputs = tokenizer([text], return_tensors="pt", padding=True)
translated = model.generate(**inputs)
result = tokenizer.decode(translated[0], skip_special_tokens=True)
print(result)

c:\Users\Administrator\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 27415.84it/s]


Xin chào, anh khỏe không?


# Build Model

In [4]:
from omegaconf import OmegaConf
from transformers import MarianMTModel, MarianTokenizer


class Translator:
    def __init__(self, config_path):
        self.config = OmegaConf.load(config_path)
        print(f"[INFO] Loading model: {self.config.model_path}")
        self.tokenizer = MarianTokenizer.from_pretrained(self.config.model_path)
        self.model     = MarianMTModel.from_pretrained(self.config.model_path)
        print("[INFO] Model loaded successfully!")

    def __call__(self, message: str) -> str:
        if not message or not message.strip():
            raise ValueError("Message cannot be empty")
        inputs = self.tokenizer(
            [message], return_tensors="pt",
            padding=True, truncation=True
        )
        translated_ids = self.model.generate(**inputs)
        return self.tokenizer.decode(translated_ids[0], skip_special_tokens=True)

# Initialize Model

In [5]:
translator = Translator("./config.yaml")

[INFO] Loading model: Helsinki-NLP/opus-mt-en-vi


Loading weights: 100%|██████████| 258/258 [00:00<00:00, 14673.35it/s]


[INFO] Model loaded successfully!


In [6]:
translator("Artificial intelligence is changing the world.")

'Tình báo đang làm thay đổi thế giới.'

# Initialize API

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)


@app.get('/')
async def root():
    return {
        "service": "English → Vietnamese Translation API",
        "model": "Helsinki-NLP/opus-mt-en-vi",
        "endpoints": {
            "GET /": "System info",
            "GET /health": "Health check",
            "GET /translate": "Translate English text to Vietnamese"
        }
    }


@app.get('/health')
async def health():
    return {
        "status": "ok",
        "model": "Helsinki-NLP/opus-mt-en-vi",
        "task": "translation_en_to_vi"
    }


## TODO
@app.get('/translate')
async def translate(message: str):
    if not message or not message.strip():
        raise HTTPException(status_code=422, detail="'message' không được để trống")
    if len(message) > 512:
        raise HTTPException(status_code=422, detail="'message' không được vượt quá 512 ký tự")
    try:
        result = translator(message)
        return {
            "source_language": "English",
            "target_language": "Vietnamese",
            "original_text": message,
            "translated_text": result
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Lỗi dịch thuật: {str(e)}")
## END TODO


import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("Server started on http://0.0.0.0:8000")

Server started on http://0.0.0.0:8000


INFO:     Started server process [21056]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


# Call Local API

In [ ]:
import requests

API_URL = "http://127.0.0.1:8000/translate"
params  = {"message": "Hello, I'm fine thank you"}

response = requests.get(API_URL, params=params)
print(response.json())


INFO:     127.0.0.1:61005 - "GET /translate?message=Hello%2C+I%27m+fine+thank+you HTTP/1.1" 200 OK
{'source_language': 'English', 'target_language': 'Vietnamese', 'original_text': "Hello, I'm fine thank you", 'translated_text': 'Xin chào, tôi ổn cảm ơn bạn'}


# Call Public API

In [ ]:
import requests

# Thay URL bằng link pinggy của bạn sau khi chạy tunnel
API_URL = "https://your-tunnel-url.a.free.pinggy.link/translate"
params  = {"message": "Hello, how are you?"}

response = requests.get(API_URL, params=params)
print(response.json())

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))